In [16]:
import os
import glob
import random
import pandas as pd
import numpy as np
import torch  
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from numba import njit, float64, int64, uint64,types
from numba.typed import Dict
from tqdm import tqdm
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

In [17]:
# ==========================================
# 1. 설정 (Configuration)
# ==========================================
# 실제 데이터가 있는 경로로 수정하세요
BASE_PATH = "C:/Users/user/Desktop/IDS_masters/Car_Hacking_Challenge_Dataset_rev20Mar2021/0_Preliminary/1_Submission"
PT_SAVE_PATH = "C:/Users/user/Desktop/IDS_masters/dataset/training_dataset_yj.pt"
CSV_SAVE_PATH = "C:/Users/user/Desktop/IDS_masters/dataset/training_dataset_yj.csv"
WINDOW_SIZE = 128
STRIDE = 64  # 50% Overlap

# 공격 라벨 정의
ATTACK_LABELS = {
    "Normal": 0,
    "Flooding": 1,
    "Fuzzing": 2,
    "Replay": 3,
    "Spoofing": 4
}
LABEL_MAP = {
    "Normal": 0,
    "Flooding": 1,   # 원본 명칭
    "DoS": 1,        # 혹시 나중에 DoS라는 문자열도 들어오면 같이 1로 처리
    "Fuzzing": 2,
    "Replay": 3,
    "Spoofing": 4,
}

FEATURE_NAMES = [
    "1.IAT", "2. Is_Zero", "3. Payload_Ent", "4. Complexity", 
    "5. Ham_Rate", "6. Freq", "7. Continuity", "8. Diff_Ent", "9. ID_Ent", "10.", "11.","12"
]


In [18]:
@njit
def popcount64(x):
    c = 0
    v = int64(x)
    while v:
        v &= v - int64(1)
        c += 1
    return c


@njit
def pack_payload_u64_dlc(row, dlc):
    v = uint64(0)
    d = dlc
    if d < 0:
        d = 0
    if d > 8:
        d = 8
    for i in range(d):
        v |= uint64(row[i]) << (i * 8)
    return v


@njit(fastmath=True)
def calculate_features(timestamps, can_ids, dlcs, payloads):

    n = len(timestamps)

    # feature index
    # 0  is_cid0
    # 1  dlc/8
    # 2  log1p(rel_change)/5
    # 3  log1p(ent * rel_change)
    # 4  freq_local
    # 5  id_ent
    # 6  streak_ratio_fixed
    # 7  norm_by_win_mean
    # 8  freq_over_top1_log
    # 9  top1_share_fixed
    # 10 idx_gap_cv
    # 11 dominance_ratio

    features = np.zeros((n, 12), dtype=np.float64)

    last_time_map    = Dict.empty(key_type=types.int64, value_type=types.float64)
    last_payload_map = Dict.empty(key_type=types.int64, value_type=types.uint64)

    local_cnt_map = Dict.empty(key_type=types.int64, value_type=types.float64)
    id_ham_ema    = Dict.empty(key_type=types.int64, value_type=types.float64)
    streak_map    = Dict.empty(key_type=types.int64, value_type=types.int64)

    last_pos_map = Dict.empty(key_type=types.int64, value_type=types.int64)
    gap_n_map    = Dict.empty(key_type=types.int64, value_type=types.int64)
    gap_mean_map = Dict.empty(key_type=types.int64, value_type=types.float64)
    gap_M2_map   = Dict.empty(key_type=types.int64, value_type=types.float64)

    alpha_ham = 0.05
    beta_streak = 0.2
    eps = 1e-9
    W = 128.0

    top1_id = int64(-1)
    top2_id = int64(-1)
    top1_cnt = 0.0
    top2_cnt = 0.0

    for i in range(n):

        if (i % 128) == 0:
            local_cnt_map.clear()
            streak_map.clear()
            last_pos_map.clear()
            gap_n_map.clear()
            gap_mean_map.clear()
            gap_M2_map.clear()

            top1_id = int64(-1)
            top2_id = int64(-1)
            top1_cnt = 0.0
            top2_cnt = 0.0

        cid = int64(can_ids[i])
        dlc = int64(dlcs[i])
        row = payloads[i]
        pos = int64(i % 128)

        if cid in last_time_map:
            curr_iat = timestamps[i] - last_time_map[cid]
            if curr_iat < 0.0:
                curr_iat = 0.0
        last_time_map[cid] = timestamps[i]

        if dlc < 0:
            dlc = 0
        elif dlc > 8:
            dlc = 8

        cur_bytes = pack_payload_u64_dlc(row, dlc)

        h_dist = 0.0
        rel_change = 0.0
        streak = int64(0)

        if cid in last_payload_map:
            diff_bits = cur_bytes ^ last_payload_map[cid]
            h_dist = float64(popcount64(diff_bits))

            if diff_bits == uint64(0):
                streak = streak_map.get(cid, int64(0)) + int64(1)
            else:
                streak = int64(0)
            streak_map[cid] = streak

            if cid in id_ham_ema:
                avg_h = id_ham_ema[cid]
                rel_change = h_dist / (avg_h + 0.1)
                id_ham_ema[cid] = (1.0 - alpha_ham) * avg_h + alpha_ham * h_dist
            else:
                rel_change = 1.0
                id_ham_ema[cid] = h_dist
        else:
            streak_map[cid] = int64(0)

        last_payload_map[cid] = cur_bytes

        ent = 0.0
        if dlc > 0:
            p_counts = np.zeros(256, dtype=np.int64)
            for k in range(dlc):
                p_counts[row[k]] += 1
            for c in p_counts:
                if c > 0:
                    p = c / float64(dlc)
                    ent -= p * np.log(p)

        id_ent = 0.0
        if i >= 127:
            win_id_counts = Dict.empty(key_type=types.int64, value_type=types.float64)
            for j in range(i - 127, i + 1):
                wid = int64(can_ids[j])
                win_id_counts[wid] = win_id_counts.get(wid, 0.0) + 1.0
            for k_id in win_id_counts:
                pk = win_id_counts[k_id] / W
                id_ent -= pk * np.log(pk + 1e-9)
            id_ent = id_ent / 4.85

        cnt = local_cnt_map.get(cid, 0.0) + 1.0
        local_cnt_map[cid] = cnt
        freq_local = cnt / W

        num_unique = float64(len(local_cnt_map))
        total_seen = float64(pos + 1)
        mean_count = total_seen / (num_unique + eps)
        norm_by_win_mean = cnt / (mean_count + eps)

        if cid == top1_id:
            top1_cnt = cnt
        elif cid == top2_id:
            top2_cnt = cnt

        if cnt > top1_cnt + 1e-12:
            if cid != top1_id:
                top2_id = top1_id
                top2_cnt = top1_cnt
                top1_id = cid
                top1_cnt = cnt
        elif cnt > top2_cnt + 1e-12 and cid != top1_id:
            top2_id = cid
            top2_cnt = cnt

        freq_over_top1_log = np.log((cnt + eps) / (top1_cnt + eps))
        top1_share_fixed = top1_cnt / W
        dominance_ratio = (top1_cnt - top2_cnt) / (top1_cnt + eps)
        streak_ratio_fixed = float64(streak) / W

        idx_gap_cv = 0.0
        if cid in last_pos_map:
            gap = pos - last_pos_map[cid]
            if gap <= 0:
                gap = int64(1)

            if cid in gap_n_map:
                gn = gap_n_map[cid] + int64(1)
                gmean = gap_mean_map[cid]
                gM2 = gap_M2_map[cid]

                x = float64(gap)
                delta = x - gmean
                gmean = gmean + delta / float64(gn)
                delta2 = x - gmean
                gM2 = gM2 + delta * delta2

                gap_n_map[cid] = gn
                gap_mean_map[cid] = gmean
                gap_M2_map[cid] = gM2
            else:
                gap_n_map[cid] = int64(1)
                gap_mean_map[cid] = float64(gap)
                gap_M2_map[cid] = 0.0

            gn = gap_n_map[cid]
            gmean = gap_mean_map[cid]
            gM2 = gap_M2_map[cid]
            if gn >= 2:
                var = gM2 / float64(gn - 1)
                if var < 0.0:
                    var = 0.0
                gstd = np.sqrt(var)
                idx_gap_cv = gstd / (gmean + eps)

        last_pos_map[cid] = pos

        features[i, 0] = 1.0 if cid == 0 else 0.0
        features[i, 1] = float64(dlc) / 8.0
        features[i, 2] = np.log1p(rel_change) / 5.0
        features[i, 3] = np.log1p(ent * rel_change)
        features[i, 4] = freq_local
        features[i, 5] = id_ent
        features[i, 6] = streak_ratio_fixed
        features[i, 7] = norm_by_win_mean
        features[i, 8] = freq_over_top1_log
        features[i, 9] = top1_share_fixed
        features[i, 10] = idx_gap_cv
        features[i, 11] = dominance_ratio

    return features


In [19]:
def make_windows_from_stream(features, labels, window_size=128, stride=64):
    """
    features: (N, F)
    labels:   (N,)  또는 (N, ) per packet label
    return:
      Xw: (Nwin, F, L)
      yw: (Nwin, L)
    """
    N, F = features.shape
    L = window_size

    nwin = 1 + (N - L) // stride
    Xw = np.zeros((nwin, F, L), dtype=np.float32)
    yw = np.zeros((nwin, L), dtype=np.int64)

    w = 0
    for start in range(0, N - L + 1, stride):
        end = start + L
        # (L, F) -> (F, L)
        Xw[w] = features[start:end].T.astype(np.float32)
        yw[w] = labels[start:end].astype(np.int64)
        w += 1

    return Xw, yw

In [20]:
# ==========================================
# 2. 헬퍼 함수 (ID 파싱, Payload 파싱)
# ==========================================
def parse_id(id_val):
    if isinstance(id_val, str):
        try:
            return int(id_val, 16)
        except:
            return 0
    return int(id_val)

def parse_payload_str(s, max_len=8):
    """'00 00 A1 ...' 형태의 문자열을 길이 8의 리스트로 변환"""
    parts = str(s).split()
    vals = []
    for p in parts:
        if p != "":
            try:
                vals.append(int(p, 16))
            except:
                pass
    
    if len(vals) < max_len:
        vals += [0] * (max_len - len(vals))
    return vals[:max_len]

In [21]:
def main():
    # 1. CSV 파일 목록
    csv_files = glob.glob(os.path.join(BASE_PATH, "*.csv"))
    if not csv_files:
        print(f"[ERROR] 해당 경로에 CSV 파일이 없습니다: {BASE_PATH}")
        return

    print(f"[INFO] 발견된 파일: {len(csv_files)}개")
    for f in csv_files:
        print("   -", os.path.basename(f))

    # 2. CSV 통합
    df_list = []
    for file in csv_files:
        print(f"[READING] {os.path.basename(file)} 읽는 중...")
        temp_df = pd.read_csv(file, header=0)
        df_list.append(temp_df)

    full_df = pd.concat(df_list, axis=0, ignore_index=True)
    print(f"[INFO] 통합 완료. 총 패킷 수: {len(full_df)}")

    # 3. 라벨 정리 (Flooding → DoS 이름 통일)
    full_df["SubClass"] = full_df["SubClass"].astype(str).str.strip()
    full_df["SubClass"] = full_df["SubClass"].replace("Flooding", "DoS")

    # 패킷 단위 문자열 라벨
    raw_labels = full_df["SubClass"].astype(str).values

    # 4. 피처 계산에 필요한 컬럼 → numpy
    timestamps = full_df["Timestamp"].astype(np.float64).to_numpy()
    can_ids = full_df["Arbitration_ID"].apply(parse_id).astype(np.int64).to_numpy()
    dlcs = full_df["DLC"].astype(np.int64).to_numpy()
    payload_array = np.vstack(
        full_df["Data"].apply(parse_payload_str).values
    ).astype(np.uint8)

    print("[INFO] 통합 데이터 피처 계산 중...")
    
    # all_features.shape = (패킷 수, 9)

    # 5. ===== 패킷 레벨 CSV 저장 =====
    packet_labels_int = np.vectorize(LABEL_MAP.get)(raw_labels).astype(np.int64)

    feat_stream = calculate_features(timestamps, can_ids, dlcs,payload_array)
    X_np, y_np = make_windows_from_stream(feat_stream, packet_labels_int, window_size=128, stride=64)

    df_packet = pd.DataFrame(feat_stream, columns=FEATURE_NAMES)
    df_packet["Label_Int"] = packet_labels_int
    df_packet["Label_Str"] = raw_labels
    df_packet.to_csv(CSV_SAVE_PATH, index=False)
    print(f"[DONE] .csv 패킷 단위 저장 완료: {CSV_SAVE_PATH}")
    print(f"       형태: {df_packet.shape} (행: 패킷 수, 열: 특징+라벨)")

    np.savez(
    "C:/Users/user/Desktop/IDS_masters/dataset/carchallenge_0305.npz",
    X=X_np.astype(np.float32),
    y=y_np.astype(np.int64)
    )

    print(f" Saved dataset")

if __name__ == "__main__":
    main()

[INFO] 발견된 파일: 2개
   - Pre_submit_D.csv
   - Pre_submit_S.csv
[READING] Pre_submit_D.csv 읽는 중...
[READING] Pre_submit_S.csv 읽는 중...
[INFO] 통합 완료. 총 패킷 수: 3752046
[INFO] 통합 데이터 피처 계산 중...
[DONE] .csv 패킷 단위 저장 완료: C:/Users/user/Desktop/IDS_masters/dataset/training_dataset_yj.csv
       형태: (3752046, 14) (행: 패킷 수, 열: 특징+라벨)
 Saved dataset


In [22]:
import numpy as np

Path = "C:/Users/User/Desktop/IDS_masters/dataset/carchallenge_test_0226.npz"

data = np.load(Path)

X = data["X"]
y = data["y"]

print(f"x shape: {X.shape}")
print(f"y shape: {y.shape}")

unique, counts = np.unique(y, return_counts=True)
print(unique)
print(counts)

uniq_dict = dict(zip(unique, counts))
print(f"클래스 별 데이터 수: {uniq_dict}")

x shape: (51750, 12, 128)
y shape: (51750, 128)
[0 1 2 3 4]
[6025184  308360  179758   95186   15512]
클래스 별 데이터 수: {np.int64(0): np.int64(6025184), np.int64(1): np.int64(308360), np.int64(2): np.int64(179758), np.int64(3): np.int64(95186), np.int64(4): np.int64(15512)}
